# FreshCart ELT: bronze -> silver -> quarantine -> gold

Set `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD`, `SNOWFLAKE_DATABASE`, and `SNOWFLAKE_SCHEMA` before running. The SQL files in `sql/` are idempotent and can also be run in Snowsight.

In [ ]:
from pathlib import Path
import os
import snowflake.connector

con = snowflake.connector.connect(
    account=os.environ['SNOWFLAKE_ACCOUNT'], user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'], database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ.get('SNOWFLAKE_SCHEMA', 'PUBLIC'), warehouse=os.environ['SNOWFLAKE_WAREHOUSE'])
for sql_file in sorted(Path('sql').glob('*.sql')):
    print(sql_file.name)
    con.cursor().execute_stream(sql_file.read_text())

The final queries below are the evidence cells for the assignment: bronze row count, clean/quarantine counts, deduplicated order count, gold revenue and points, and quality checks.

In [ ]:
checks = {
    'raw_rows': 'SELECT COUNT(*) FROM bronze.orders_raw',
    'clean_rows': 'SELECT COUNT(*) FROM silver.orders_clean',
    'quarantine_rows': 'SELECT COUNT(*) FROM silver.orders_quarantine',
    'gold_orders': 'SELECT COUNT(*) FROM gold.orders',
    'gold_totals': 'SELECT SUM(amount_inr), SUM(loyalty_points) FROM gold.orders',
}
for name, query in checks.items():
    print(name, con.cursor().execute(query).fetchone())